## **Cleaning and Target Engineering**


This notebook transforms the raw FinAccess 2021 survey dataset into a clean, leakage-controlled dataset for analysis and modeling.

The main goal is to safely create the financial exclusion target variable and separate variables into:

1. target-engineering variables

2. unsafe leakage variables

3. administrative/ID variables

4. safe predictor variables

This notebook does not train models. Its purpose is to prepare a trustworthy modeling dataset for EDA, subgroup analysis, and machine learning.

In [1]:
# Import Libraries
import pandas as pd
import numpy as np
import json
from pathlib import Path

In [2]:
BASE_DIR = Path("..")

RAW_DATA_PATH = BASE_DIR / "Data" / "Raw" / "finaccess_2021_microdata.xlsx"
PROCESSED_DIR = BASE_DIR / "Data" / "Processed"
ARTIFACTS_DIR = BASE_DIR / "artifacts" / "metrics"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

#### **Load Raw Data**
We reload the raw dataset instead of depending on objects from Data Understanding Notebook. This keeps the notebook reproducible and allows it to run independently.

In [3]:
df = pd.read_excel(RAW_DATA_PATH, sheet_name="Dataset")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 22024
Columns: 2332


#### **Target Engineering**
The target variable represents whether a respondent is financially excluded.

Based on data understanding Notebook, the column excluded_informal_banked2022 contains three financial access categories:

 - Banked

-  Other Formal

-  Excluded

For the first modeling version, this project uses binary classification:

- 1 = financially excluded
- 0 = not financially excluded

Banked and Other Formal are grouped as not excluded because they indicate some form of formal financial access.

In [4]:
target_source_col = "excluded_informal_banked2022"

df["financially_excluded"] = df[target_source_col].map({
    "Excluded": 1,
    "Banked": 0,
    "Other Formal": 0
})

In [5]:
#validate target
df["financially_excluded"].value_counts(dropna=False)

0    17430
1     4594
Name: financially_excluded, dtype: int64

In [6]:
target_distribution = (
    df["financially_excluded"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
    .reset_index()
)

target_distribution.columns = ["financially_excluded", "percentage"]
target_distribution

,financially_excluded,percentage
0,0,79.14
1,1,20.86


In [7]:
target_distribution.to_csv(
    ARTIFACTS_DIR / "02_target_distribution.csv",
    index=False
)

The target is imbalanced. Financially excluded respondents are the minority class.

This means accuracy alone will not be enough during modeling. Later notebooks should prioritize recall, precision, F1-score, ROC-AUC, and confusion matrix interpretation.